# eph_00 — Single-unit inspection

> **This notebook requires Code Ocean.** It uses session-level intermediate
> files (spike times, kinematics, trial events) that are not available locally.
> Run it inside the capsule where `units_with_spikes` and session directories
> are populated.

Interactive single-unit visualization for quality checking and exploratory analysis.

**Pipeline:**
1. Load session data from Code Ocean intermediates
2. Select a session + unit by index, session string, or unit_id
3. Single-trial kinematics + spike overlay (`plot_single_trial`)
4. Raster + PSTH aligned to behavioral events

## 1. Setup

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
from pathlib import Path

# This notebook runs on Code Ocean only.
SCRATCH   = Path("/root/capsule/scratch")
DATA_ROOT = Path("/root/capsule/data")

FIG_DIR  = SCRATCH / "figures" / "eph_00_inspection"
SAVE_FIG = False

## 2. Data loading

In [ ]:
from data_loading import (
    load_session_quality_filter, filter_ephys_units, load_units_with_spike_times,
)
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import (
    load_intermediate_data, find_session_dir,
)
import pickle

base_dirs = [SCRATCH / "session_analysis_mlk"]
filtered_session_paths = load_session_quality_filter(base_dirs)

with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
    combined_ephys_data = pickle.load(f)

filtered_ephys    = filter_ephys_units(combined_ephys_data, filtered_session_paths)
ROOT_SCRATCH      = str(DATA_ROOT / "LC-NE_scratch_data_1")
units_with_spikes = load_units_with_spike_times(filtered_ephys, ROOT_SCRATCH)
print(f"units_with_spikes: {units_with_spikes.shape}")

## 3. Session bundle config

In [ ]:
from ephys_utils import AnalysisConfig

cfg = AnalysisConfig(
    align_key="goCue",
    count_window_s=(0.0, 0.2),
    baseline_window_s=(-1.0, 0.0),
    min_trials_per_group=20,
)

## 4. Helper: load one session + unit

`load_example_session_and_unit` picks a row from `units_with_spikes`,
loads per-session kinematics / trial data, and returns a dict for inspection.

In [ ]:
def load_example_session_and_unit(
    units_with_spikes: pd.DataFrame,
    cfg: AnalysisConfig,
    idx: int = 0,
) -> dict:
    """
    Convenience loader for exploratory analysis.

    Picks one row from units_with_spikes by integer index, loads the
    session-level intermediate data, and converts spike times to session time.

    Returns
    -------
    dict with keys: session, unit_id, spikes_session_time,
                    movs, kins, trials, licks, events
    """
    row     = units_with_spikes.iloc[idx]
    session = row.session
    unit_id = row.unit_id

    sdir = find_session_dir(session, roots=base_dirs)
    data = load_intermediate_data(sdir)

    evnts = data["events"]
    # Session offset: first go-cue timestamp anchors t=0
    session_offset = evnts.loc[
        evnts["event"] == "goCue_start_time", "raw_timestamps"
    ].iloc[0]

    spikes_session_time = np.asarray(row.spike_times, dtype=float) - session_offset

    return {
        "session":              session,
        "unit_id":              unit_id,
        "spikes_session_time":  spikes_session_time,
        "movs":                 data["movs"],
        "kins":                 data["kins"],
        "trials":               data["trials"],
        "licks":                data["licks"],
        "events":               evnts,
    }

## 5. Select a unit to inspect

In [ ]:
# ---- Unit selection ----
# Option A: by row index (0 = first unit in filtered table)
EXAMPLE_IDX = 0

# Option B: by session + unit_id (uncomment to use)
# TARGET_SESSION = "behavior_791691_2025-06-25_14-06-10"
# TARGET_UNIT    = 570
# EXAMPLE_IDX    = units_with_spikes.index[
#     (units_with_spikes["session"] == TARGET_SESSION) &
#     (units_with_spikes["unit_id"] == TARGET_UNIT)
# ][0]

example = load_example_session_and_unit(units_with_spikes, cfg, idx=EXAMPLE_IDX)

session = example["session"]
unit_id = example["unit_id"]
spikes  = example["spikes_session_time"]
movs    = example["movs"]
kins    = example["kins"]
trials  = example["trials"]
events  = example["events"]

print(f"Session: {session}")
print(f"Unit:    {unit_id}")
print(f"Spikes:  {len(spikes):,}")
print(f"Trials:  {len(trials):,}")

## 6. Single-trial kinematics + spike overlay

`plot_single_trial` shows tongue position (y) for a window around the go cue.

In [ ]:
def plot_single_trial(
    trials: pd.DataFrame,
    kins: pd.DataFrame,
    trial_n: int,
    pre_go: float = 2.0,
    post_go: float = 3.0,
    y_col: str = "y",
    go_col: str = "goCue_start_time_in_session",
) -> plt.Figure:
    """
    Plot tongue kinematics (y position) for one trial, relative to go cue.

    Parameters
    ----------
    trials : DataFrame
        Must have `trial` and `go_col` columns.
    kins : DataFrame
        Must have `trial`, `time_in_session`, and `y_col` columns.
    trial_n : int
        Trial number to plot.
    pre_go, post_go : float
        Time window (s) before and after go cue.

    Returns
    -------
    matplotlib Figure
    """
    t0 = trials.loc[trials["trial"] == trial_n, go_col].iloc[0]

    mask = (
        (kins["trial"] == trial_n)
        & (kins["time_in_session"] >= t0 - pre_go)
        & (kins["time_in_session"] <= t0 + post_go)
    )
    k     = kins.loc[mask]
    t_rel = k["time_in_session"] - t0

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(t_rel, k[y_col], alpha=0.4, color=PALETTE["neutral"])
    ax.scatter(t_rel, k[y_col], s=10, color=PALETTE["neutral"])
    ax.axvline(0, linestyle="--", linewidth=0.8, color=PALETTE["neg"], label="go cue")
    ax.set_xlim(-pre_go, post_go)
    ax.set_xlabel("Time from go cue (s)")
    ax.set_ylabel("Position (pix)")
    ax.set_title(f"Trial {trial_n}")
    ax.legend(frameon=False, fontsize=8)
    style_ax(ax)
    plt.tight_layout()
    return fig

In [ ]:
fig = plot_single_trial(trials, kins, trial_n=30, pre_go=0.2, post_go=0.45)
save_fig(fig, "example_trial", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 7. Raster + PSTH aligned to behavioral event

Uses `make_rp_and_events` + `plot_psth` from `aind_dynamic_foraging_behavior_video_analysis`.

In [ ]:
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import (
    make_rp_and_events, compute_psth, smooth_vector, plot_psth,
)

# --- Raster + PSTH parameters ---
ALIGN_BY = "goCue"      # event to align raster
SORT_BY  = "firstMove"  # event to sort trials by
PRE_S    = 1.0          # window before align event (s)
POST_S   = 2.0          # window after align event (s)
BIN_SIZE = 0.001        # raster bin width (s)

In [ ]:
row = units_with_spikes.query(
    "session == @session and unit_id == @unit_id"
).iloc[0]
spikes_abs = np.array(row["spike_times"], dtype=float)

bundle         = bundle_cache[session]
Ev             = bundle["Ev"]
align_times    = bundle["align_times"]
trial_features = bundle["trial_features"]

event_dicts = {
    "goCue":     Ev["goCue_start_time"].values,
    "firstMove": Ev["firstMove_start_time"].values,
    "firstLick": Ev["firstLick_start_time"].values,
    "reward":    Ev["reward_delivery_time"].values,
}

rp, events = make_rp_and_events(
    spikes=spikes_abs,
    trials=trials,
    event_dicts=event_dicts,
    events_to_plot=["goCue", "firstMove", "firstLick", "reward"],
    align_by=ALIGN_BY,
    sort_by=SORT_BY,
    pre=PRE_S, post=POST_S, bin_size=BIN_SIZE,
)

In [ ]:
fig, (ax_raster, ax_psth) = plt.subplots(
    2, 1, figsize=(8, 7), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)

rp.plot_raster(ax=ax_raster, spike_color="black")
rp.add_events(ax_raster, events)
ax_raster.set_title(f"Unit {unit_id}  |  align: {ALIGN_BY}   sort: {SORT_BY}")
ax_raster.legend(loc="upper left", title="Events", fontsize=7)

a, b = cfg.count_window_s
ax_raster.axvspan(a, b, alpha=0.08, linewidth=0)

psth, _  = compute_psth(rp.raster, bin_size=rp.bin_size)
psth_sm  = smooth_vector(psth, bin_size=rp.bin_size, sigma=0.025)
plot_psth(rp.bins, psth, psth_sm, ax=ax_psth, label="PSTH")
ax_psth.axvspan(a, b, alpha=0.08, linewidth=0)
ax_psth.set_xlabel(f"Time from {ALIGN_BY} (s)")

plt.tight_layout()
save_fig(fig, f"raster_psth_{ALIGN_BY}_sort_{SORT_BY}", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 8. Alternative alignment: first lick

Re-align to `firstLick`, sort by `goCue` for comparison.

In [ ]:
ALT_ALIGN = "firstLick"
ALT_SORT  = "goCue"

rp2, events2 = make_rp_and_events(
    spikes=spikes_abs,
    trials=trials,
    event_dicts=event_dicts,
    events_to_plot=["goCue", "firstMove", "firstLick", "reward"],
    align_by=ALT_ALIGN,
    sort_by=ALT_SORT,
    pre=1.0, post=2.0, bin_size=BIN_SIZE,
)

fig, (ax_r2, ax_p2) = plt.subplots(
    2, 1, figsize=(8, 7), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)
rp2.plot_raster(ax=ax_r2, spike_color="black")
rp2.add_events(ax_r2, events2)
ax_r2.set_title(f"Unit {unit_id}  |  align: {ALT_ALIGN}   sort: {ALT_SORT}")
ax_r2.legend(loc="upper left", title="Events", fontsize=7)

psth2, _  = compute_psth(rp2.raster, bin_size=rp2.bin_size)
psth2_sm  = smooth_vector(psth2, bin_size=rp2.bin_size, sigma=0.025)
plot_psth(rp2.bins, psth2, psth2_sm, ax=ax_p2, label="PSTH")
ax_p2.set_xlabel(f"Time from {ALT_ALIGN} (s)")

plt.tight_layout()
save_fig(fig, f"raster_psth_{ALT_ALIGN}_sort_{ALT_SORT}", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()